# Inspect VDS

In [ ]:
# %%configure -f
# {
#   "driverMemory": "45G",
#   "conf": {
#     "spark.driver.memoryOverhead": "8G",
#     "spark.yarn.driver.memoryOverhead": "8G",
#     "spark.driver.maxResultSize": "4G",
#     "spark.speculation": "false"
#   }
# }

In [1]:
%%configure -f
{
    "driverMemory": "45G"
}

In [2]:
# Import and initiate HAIL
import hail as hl
hl.init(sc,log='/tmp/hail.log')

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
3,application_1755050209389_0004,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

pip-installed Hail requires additional configuration options in Spark referring
  to the path to the Hail Python module directory HAIL_DIR,
  e.g. /path/to/python/site-packages/hail:
    spark.jars=HAIL_DIR/backend/hail-all-spark.jar
    spark.driver.extraClassPath=HAIL_DIR/backend/hail-all-spark.jar
    spark.executor.extraClassPath=./hail-all-spark.jarRunning on Apache Spark version 3.5.2-amzn-1
SparkUI available at http://ip-192-168-103-103.ap-southeast-1.compute.internal:41193
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.134-952ae203dbbe
LOGGING: writing to /tmp/hail.log

In [3]:
from pprint import pprint
pprint(dict(hl.spark_context().getConf().getAll()))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

{'spark.app.attempt.id': '1',
 'spark.app.id': 'application_1755050209389_0004',
 'spark.app.name': 'livy-session-3',
 'spark.app.startTime': '1755054016216',
 'spark.app.submitTime': '1755053993241',
 'spark.blacklist.decommissioning.enabled': 'true',
 'spark.blacklist.decommissioning.timeout': '1h',
 'spark.decommissioning.timeout.threshold': '20',
 'spark.default.parallelism': '16',
 'spark.driver.defaultJavaOptions': "-XX:OnOutOfMemoryError='kill -9 %p'",
 'spark.driver.extraClassPath': '/usr/local/lib/python3.9/site-packages/hail/backend/hail-all-spark.jar:/usr/lib/hadoop-lzo/lib/*:/usr/lib/hadoop/hadoop-aws.jar:/usr/share/aws/aws-java-sdk/*:/usr/share/aws/emr/emrfs/conf:/usr/share/aws/emr/emrfs/lib/*:/usr/share/aws/emr/emrfs/auxlib/*:/usr/share/aws/emr/goodies/lib/emr-spark-goodies.jar:/usr/share/aws/emr/security/conf:/usr/share/aws/emr/security/lib/*:/usr/share/aws/hmclient/lib/aws-glue-datacatalog-spark-client.jar:/usr/share/java/Hive-JSON-Serde/hive-openx-serde.jar:/usr/share/

In [4]:
# source

vds_prefix = 's3://precise-scratch/goypav/1KG/VDS/'

vds_uri = vds_prefix + '1000genomes_combined_batch1_2_3_4.bf2-tr500k-sp1k.n3205.vds'

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [5]:
vds = hl.vds.read_vds(vds_uri)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [6]:
vds.reference_data.describe()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

----------------------------------------
Global fields:
    'ref_block_max_length': int32
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
----------------------------------------
Entry fields:
    'LEN': int32
    'DP': int32
    'GQ': int32
    'ICNT': array<int32>
    'MIN_DP': int32
    'SPL': array<int32>
    'LGT': call
    'LAD': array<int32>
    'END': int32
----------------------------------------
Column key: ['s']
Row key: ['locus']
----------------------------------------

In [7]:
hl.eval(vds.reference_data.ref_block_max_length)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

151393

In [8]:
vds.variant_data.describe()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
----------------------------------------
Entry fields:
    'LA': array<int32>
    'LGT': call
    'LAD': array<int32>
    'LPL': array<int32>
    'RGQ': int32
    'gvcf_info': struct {
        DB: bool, 
        FS: float64, 
        FractionInformativeReads: float64, 
        LOD: float64, 
        MQ: float64, 
        MQRankSum: float64, 
        QD: float64, 
        R2_5P_bias: float64, 
        ReadPosRankSum: float64, 
        SOR: float64
    }
    'AF': array<float64>
    'DP': int32
    'F1R2': array<int32>
    'F2R1': array<int32>
    'GP': array<float64>
    'GQ': int32
    'ICNT': array<int32>
    'MB': array<int32>
    'MIN_DP': int32
    'PRI': array<float64>
    'PS': int32
    'SB': array<int32>
    'SPL': array<int32

In [9]:
# Count rows/cols in the variant_data MT
print(f"Reference genome: {vds.variant_data.locus.dtype.reference_genome.name}")
print(f"Number of samples: {vds.n_samples()}")
print(f"Number of variant partitions: {vds.variant_data.n_partitions()}")
print(f"Total number of variants: {vds.variant_data.count_rows():,}")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Reference genome: GRCh38
Number of samples: 3205
Number of variant partitions: 5767
Total number of variants: 156,228,032

In [12]:

def vds_nano_qc(vds, p=0.003, coalesce=64, tmp_path=None, seed=1):
    """
    Lightweight QC for small clusters (no densify).
    - p: row sampling fraction (e.g. 0.003 = 0.3%)
    - coalesce: reduce partitions without shuffle
    - tmp_path: optional checkpoint path to speed repeats
    """
    hl.set_global_seed(seed)

    # Keep only row keys (locus, alleles) and drop entry fields
    vd = vds.variant_data.select_rows().select_entries()

    # Fewer tasks for an 8-CPU cluster
    if coalesce:
        vd = vd.naive_coalesce(coalesce)

    # Optional: cache slim view for repeated QC runs
    if tmp_path:
        vd = vd.checkpoint(tmp_path, _read_if_exists=True)

    # Smoke test – should start tasks immediately
    vd.rows().show(5)

    # Tiny random sample for quick aggregates
    sampled = vd.filter_rows(hl.rand_bool(p))

    agg = sampled.aggregate_rows(hl.struct(
        total = hl.agg.count(),
        bi    = hl.agg.count_where(hl.len(sampled.alleles) == 2),

        # Type mix (based on first alt)
        snp = hl.agg.count_where(hl.is_snp(sampled.alleles[0], sampled.alleles[1])),
        ins = hl.agg.count_where(hl.len(sampled.alleles[1]) > hl.len(sampled.alleles[0])),
        dele = hl.agg.count_where(hl.len(sampled.alleles[1]) < hl.len(sampled.alleles[0])),
        mnv = hl.agg.count_where(
            (hl.len(sampled.alleles[1]) == hl.len(sampled.alleles[0])) &
            (hl.len(sampled.alleles[0]) > 1)
        ),

        # Ti/Tv
        ti = hl.agg.count_where(
            hl.is_snp(sampled.alleles[0], sampled.alleles[1]) &
            hl.is_transition(sampled.alleles[0], sampled.alleles[1])
        ),
        tv = hl.agg.count_where(
            hl.is_snp(sampled.alleles[0], sampled.alleles[1]) &
            ~hl.is_transition(sampled.alleles[0], sampled.alleles[1])
        ),

        # Minimal representation
        not_minrep = hl.agg.count_where(
            (hl.min_rep(sampled.locus, sampled.alleles).locus != sampled.locus) |
            (hl.min_rep(sampled.locus, sampled.alleles).alleles != sampled.alleles)
        ),

        # Contig distribution
        contigs = hl.agg.counter(sampled.locus.contig),
    ))

    scale = int(round(1 / p))
    est_total = agg.total * scale
    est_bi    = agg.bi    * scale
    est_multi = est_total - est_bi

    print("\n≈Row-level QC (scaled from sample):")
    print(f"Estimated total variants: {est_total:,}")
    print(f"Estimated biallelic: {est_bi:,}")
    print(f"Estimated multiallelic: {est_multi:,} ({est_multi/est_total*100:.2f}%)")

    print("\nVariant type mix (approx):")
    for label, val in {
        "SNP": agg.snp, "INS": agg.ins, "DEL": agg.dele, "MNV": agg.mnv
    }.items():
        print(f"  {label}: {val*scale:,}")

    if agg.tv > 0:
        print(f"\nTi/Tv (approx): {agg.ti/agg.tv:.3f}  "
              f"(Ti ~{agg.ti*scale:,}, Tv ~{agg.tv*scale:,})")
    else:
        print("\nTi/Tv: not enough SNPs in sample")

    print(f"\nNon-minimal-representation sites (approx): {agg.not_minrep*scale:,} "
          f"({agg.not_minrep/agg.total*100:.2f}% of sampled)")

    print("\nTop contigs by count (approx):")
    for contig, cnt in sorted(agg.contigs.items(), key=lambda x: -x[1])[:10]:
        print(f"  {contig}: {cnt*scale:,}")

# Example:
# vds = hl.vds.read_vds(vds_path)
# vds_nano_qc(vds, p=0.003, coalesce=64, tmp_path="s3://YOUR-BUCKET/tmp/vd_variant_light.mt")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
vds_nano_qc(vds, p=0.003, coalesce=64, tmp_path="/tmp/vd_variant_light.mt")

In [ ]:
# check metrics
vd = vds.variant_data

# --- 1) Biallelic vs multiallelic split (no genotypes needed) ---
allele_counts = vd.aggregate_rows(hl.agg.counter(hl.len(vd.alleles)))
n_biallelic = allele_counts.get(2, 0)
n_multiallelic = sum(c for k, c in allele_counts.items() if k and k > 2)
print(f"Biallelic variants: {n_biallelic:,}")
print(f"Multiallelic variants: {n_multiallelic:,} "
      f"({(n_multiallelic/(n_biallelic+n_multiallelic))*100:.2f}% of sites)")

In [ ]:
# --- 2) Variant type breakdown (SNP / ins / del / MNV / other) ---
def var_type(alleles):
    ref, alts = alleles[0], alleles[1:]
    # classify based on first alt (cheap summary)
    alt = alts[0]
    return (hl.case()
        .when(hl.is_snp(ref, alt), "SNP")
        .when(hl.len(alt) > hl.len(ref), "INS")
        .when(hl.len(alt) < hl.len(ref), "DEL")
        .when((hl.len(alt) == hl.len(ref)) & (hl.len(ref) > 1), "MNV")
        .default("OTHER"))

type_counts = vd.aggregate_rows(hl.agg.counter(var_type(vd.alleles)))
for k in ["SNP","INS","DEL","MNV","OTHER"]:
    if k in type_counts:
        print(f"{k}: {type_counts[k]:,}")

In [ ]:
# --- 3) Ti/Tv ratio (on SNPs only; from alleles, not genotypes) ---
is_snp = hl.is_snp(vd.alleles[0], vd.alleles[1])
is_ti  = hl.is_transition(vd.alleles[0], vd.alleles[1])
counts_ti_tv = vd.filter_rows(is_snp).aggregate_rows(
    hl.struct(ti=hl.agg.count_where(is_ti), tv=hl.agg.count_where(~is_ti))
)
if counts_ti_tv.tv > 0:
    print(f"Transitions: {counts_ti_tv.ti:,}, Transversions: {counts_ti_tv.tv:,}, "
          f"Ti/Tv: {counts_ti_tv.ti / counts_ti_tv.tv:.3f}")

In [ ]:
# --- 4) Contig distribution (top 25 contigs by count) ---
contig_counts = vd.aggregate_rows(hl.agg.counter(vd.locus.contig))
# sort by count desc and print a few
for contig, cnt in sorted(contig_counts.items(), key=lambda x: -x[1])[:25]:
    print(f"{contig}: {cnt:,}")

In [ ]:
# --- 5) Left-normalization / minimal representation check ---
minrep = hl.min_rep(vd.locus, vd.alleles)
n_not_minrep = vd.aggregate_rows(
    hl.agg.count_where((minrep.locus != vd.locus) | (minrep.alleles != vd.alleles))
)
print(f"Non-minimal-representation sites: {n_not_minrep:,}")

In [ ]:
# --- 6) Optional: rsID presence rate if field exists ---
row_fields = set(vd.row_value.dtype.fields)
if "rsid" in row_fields:
    n_with_rsid = vd.aggregate_rows(hl.agg.count_where(hl.is_defined(vd.rsid) & (vd.rsid != "")))
    n_total     = vds.variant_data.count_rows()
    print(f"Sites with rsID: {n_with_rsid:,} ({n_with_rsid/n_total:.2%})")

In [ ]:
# --- 7) Optional: INFO field quick scan (e.g., MQ) if present ---
if "info" in row_fields and "MQ" in vd.info.dtype.fields:
    mq_stats = vd.aggregate_rows(hl.struct(
        mean=hl.agg.mean(vd.info.MQ),
        p5=hl.agg.approx_quantiles(vd.info.MQ, 0.05),
        p50=hl.agg.approx_quantiles(vd.info.MQ, 0.50),
        p95=hl.agg.approx_quantiles(vd.info.MQ, 0.95),
    ))
    print(f"INFO.MQ mean: {mq_stats.mean:.2f}, p5: {mq_stats.p5:.2f}, "
          f"median: {mq_stats.p50:.2f}, p95: {mq_stats.p95:.2f}")